<a href="https://colab.research.google.com/github/warthadev/warthadev-backend/blob/main/Torrent_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#@title Torrent Colab Downloader
import os
import re
import time
import json
import shutil
import subprocess
import threading
import requests

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

from IPython.display import display, HTML, Javascript, clear_output

COLAB_SAVE = "/content/Torrent"
TEMP_PATH = "/content/Temp"
META_PATH = "/content/meta"

SESSION_ID = None
BACKEND_URL = None
COLAB_CONFIG = None

POST_RETRY_COUNT = 3
POST_RETRY_DELAY = 2

_state = {
    "phase": "idle",
    "files": [],
    "current_file": "",
    "current_index": 0,
    "total_files": 0,
    "completed_files": 0,
    "failed_files": 0,
    "percent": 0,
    "current_human": "0 B",
    "total_human": "0 B",
    "speed_human": "0 B/s",
    "eta_seconds": None,
    "message": "",
    "error": "",
}

_ui_display = None


def _ensure_dirs():
    for p in [COLAB_SAVE, TEMP_PATH, META_PATH]:
        os.makedirs(p, exist_ok=True)


def _format_size(num):
    try:
        n = float(num)
    except Exception:
        return "0 B"
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024.0:
            if unit == "B":
                return f"{int(n)} B"
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} PB"


def _render_ui():
    s = dict(_state)

    phase = s["phase"]
    files = s["files"]
    total_files = s["total_files"]
    completed = s["completed_files"]
    failed = s["failed_files"]
    current_idx = s["current_index"]
    current_file = s["current_file"]
    percent = s["percent"]
    current_human = s["current_human"]
    total_human = s["total_human"]
    speed_human = s["speed_human"]
    eta_seconds = s["eta_seconds"]
    message = s["message"]
    error = s["error"]

    if phase == "idle":
        header_color = "#888"
        header_label = "WAITING FOR INPUT"
    elif phase == "fetching":
        header_color = "#2196F3"
        header_label = "FETCHING CONFIG"
    elif phase == "downloading":
        header_color = "#4CAF50"
        header_label = "DOWNLOADING"
    elif phase == "uploading":
        header_color = "#FF9800"
        header_label = "UPLOADING TO HUGGING FACE"
    elif phase == "done":
        header_color = "#00BCD4"
        header_label = "ALL DONE"
    elif phase == "error":
        header_color = "#F44336"
        header_label = "ERROR"
    else:
        header_color = "#888"
        header_label = phase.upper()

    eta_str = "--"
    if isinstance(eta_seconds, (int, float)) and eta_seconds is not None and eta_seconds >= 0:
        secs = int(eta_seconds)
        if secs < 60:
            eta_str = f"{secs}s"
        elif secs < 3600:
            eta_str = f"{secs / 60:.1f}m"
        else:
            eta_str = f"{secs / 3600:.1f}h"

    if total_files > 0:
        overall_percent = int(((completed + failed) / total_files) * 100)
    else:
        overall_percent = 0

    rows_html = ""
    if files:
        for idx, f in enumerate(files, start=1):
            name = f.get("name", "")
            size = f.get("size", 0)
            status = f.get("_status", "pending")

            if status == "done":
                badge_bg = "#4CAF50"
                badge_text = "DONE"
                row_color = "#4CAF50"
            elif status == "uploading":
                badge_bg = "#FF9800"
                badge_text = "UPLOAD"
                row_color = "#FF9800"
            elif status == "downloading":
                badge_bg = "#2196F3"
                badge_text = "DOWNLOAD"
                row_color = "#2196F3"
            elif status == "downloaded":
                badge_bg = "#00BCD4"
                badge_text = "READY"
                row_color = "#00BCD4"
            elif status == "error":
                badge_bg = "#F44336"
                badge_text = "FAILED"
                row_color = "#F44336"
            else:
                badge_bg = "#555"
                badge_text = "PENDING"
                row_color = "#888"

            size_str = _format_size(size) if size else ""

            rows_html += f"""
            <div style="display: grid; grid-template-columns: 34px 1fr 90px 80px;
                        padding: 8px 10px; border-bottom: 1px solid #2a2a2a;
                        color: {row_color}; font-family: monospace; font-size: 12px;
                        align-items: center; gap: 8px;">
                <span style="color: #888; font-size: 11px;">#{idx}</span>
                <span style="overflow: hidden; text-overflow: ellipsis;
                             white-space: nowrap;">{name}</span>
                <span style="text-align: right; color: #aaa; font-size: 11px;">{size_str}</span>
                <span style="text-align: center; background: {badge_bg}; color: #fff;
                             border-radius: 4px; font-size: 10px; padding: 2px 6px;
                             font-weight: bold;">{badge_text}</span>
            </div>
            """
    else:
        rows_html = """
        <div style="padding: 16px; text-align: center; color: #666;
                    font-family: monospace; font-size: 12px;">
            No files loaded yet
        </div>
        """

    error_html = ""
    if error:
        error_html = f"""
        <div style="margin-top: 10px; padding: 10px 14px; background: #2a1a1a;
                    border-left: 4px solid #F44336; border-radius: 6px;
                    color: #F44336; font-family: monospace; font-size: 12px;
                    word-break: break-all;">
            {error}
        </div>
        """

    message_html = ""
    if message:
        message_html = f"""
        <div style="margin-top: 10px; padding: 10px 14px; background: #1a1a2e;
                    border-left: 4px solid {header_color}; border-radius: 6px;
                    color: #e0e0e0; font-family: monospace; font-size: 12px;">
            {message}
        </div>
        """

    progress_html = ""
    if phase in ("downloading", "uploading"):
        progress_html = f"""
        <div style="margin-top: 12px; padding: 12px 14px; background: #141420;
                    border-radius: 8px; border: 1px solid #2a2a4a;">
            <div style="display: flex; justify-content: space-between;
                        align-items: center; margin-bottom: 8px;">
                <span style="color: #e0e0e0; font-family: monospace; font-size: 12px;
                             overflow: hidden; text-overflow: ellipsis;
                             white-space: nowrap; max-width: 60%;">
                    {current_file}
                </span>
                <span style="color: {header_color}; font-family: monospace;
                             font-size: 11px; font-weight: bold;">
                    [{current_idx}/{total_files}] {header_label}
                </span>
            </div>
            <div style="background: #2a2a2a; border-radius: 6px; height: 18px;
                        overflow: hidden; position: relative;">
                <div style="background: linear-gradient(90deg, {header_color}, {header_color}cc);
                            height: 100%; width: {percent}%;
                            transition: width 0.3s linear; border-radius: 6px;
                            display: flex; align-items: center; justify-content: center;
                            color: #fff; font-size: 10px; font-weight: bold;">
                    {percent}%
                </div>
            </div>
            <div style="display: flex; justify-content: space-between;
                        margin-top: 6px; color: #888; font-family: monospace;
                        font-size: 11px;">
                <span>{current_human} / {total_human}</span>
                <span>{speed_human} · ETA {eta_str}</span>
            </div>
        </div>
        """

    html = f"""
    <div style="font-family: monospace; margin: 10px 0;
                background: linear-gradient(135deg, #0f0f1e, #1a1a2e, #16213e);
                border-radius: 12px; border: 1px solid #2a2a4a; padding: 16px;
                color: #e0e0e0;">
        <div style="display: flex; justify-content: space-between;
                    align-items: center; margin-bottom: 12px;">
            <div style="color: #4CAF50; font-size: 16px; font-weight: bold;
                        letter-spacing: 1px;">
                TORRENT COLAB
            </div>
            <div style="color: {header_color}; font-size: 11px; font-weight: bold;
                        background: rgba(255,255,255,0.05); padding: 4px 10px;
                        border-radius: 20px; border: 1px solid {header_color};">
                {header_label}
            </div>
        </div>

        <div style="display: flex; gap: 16px; margin-bottom: 12px;
                    font-size: 11px; color: #888;">
            <span>Total: <b style="color: #e0e0e0;">{total_files}</b></span>
            <span>Done: <b style="color: #4CAF50;">{completed}</b></span>
            <span>Failed: <b style="color: #F44336;">{failed}</b></span>
            <span>Overall: <b style="color: #00BCD4;">{overall_percent}%</b></span>
        </div>

        <div style="background: #141420; border-radius: 8px; border: 1px solid #2a2a4a;
                    max-height: 320px; overflow-y: auto;">
            {rows_html}
        </div>

        {progress_html}
        {message_html}
        {error_html}
    </div>
    """
    return html


def _ui_update():
    global _ui_display
    html = _render_ui()
    if _ui_display is None:
        _ui_display = display(HTML(html), display_id=True)
    else:
        try:
            _ui_display.update(HTML(html))
        except Exception:
            _ui_display = display(HTML(html), display_id=True)


def _set_state(**kwargs):
    _state.update(kwargs)
    try:
        _ui_update()
    except Exception:
        pass


def _post_update(payload):
    if not SESSION_ID or not BACKEND_URL:
        return False
    url = f"{BACKEND_URL.rstrip('/')}/torrent/colab-update"
    params = {"session_id": SESSION_ID}

    for attempt in range(POST_RETRY_COUNT):
        try:
            r = requests.post(url, params=params, json=payload, timeout=15)
            if r.ok:
                return True
        except Exception:
            pass
        if attempt < POST_RETRY_COUNT - 1:
            time.sleep(POST_RETRY_DELAY * (attempt + 1))
    return False


def _fetch_config(session_id, backend_url):
    url = f"{backend_url.rstrip('/')}/torrent/colab-config"
    params = {"session_id": session_id}

    for attempt in range(POST_RETRY_COUNT):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.ok:
                return r.json()
        except Exception:
            pass
        if attempt < POST_RETRY_COUNT - 1:
            time.sleep(POST_RETRY_DELAY * (attempt + 1))
    return None


def _parse_aria2_line(line):
    out = {}
    m = re.search(r"\((\d+)%\)", line)
    if m:
        out["percent"] = int(m.group(1))
    m2 = re.search(r"([\d.]+\s*[KMGTP]?i?B)/([\d.]+\s*[KMGTP]?i?B)", line)
    if m2:
        out["current_human"] = m2.group(1).strip()
        out["total_human"] = m2.group(2).strip()
    m3 = re.search(r"DL:([\d.]+\s*[KMGTP]?i?B)", line)
    if m3:
        out["speed_human"] = m3.group(1).strip()
    m4 = re.search(r"ETA:(\S+)", line)
    if m4:
        out["eta_raw"] = m4.group(1).strip()
    return out


def _eta_to_seconds(raw):
    if not raw or raw == "--":
        return None
    if raw == "0s":
        return 0
    total = 0
    for value, unit in re.findall(r"(\d+)([hms])", raw):
        v = int(value)
        if unit == "h":
            total += v * 3600
        elif unit == "m":
            total += v * 60
        else:
            total += v
    return total


def _find_target_file(work_dir, target_name):
    if not target_name:
        return None
    target_lower = target_name.lower()
    for root, _dirs, files in os.walk(work_dir):
        for f in files:
            if f.endswith(".aria2") or f.endswith(".torrent") or f == "input.txt":
                continue
            if f.lower() == target_lower:
                return os.path.join(root, f)
    return None


def _find_largest_file(work_dir):
    largest = None
    for root, _dirs, files in os.walk(work_dir):
        for f in files:
            if f.endswith(".aria2") or f.endswith(".torrent") or f == "input.txt":
                continue
            candidate = os.path.join(root, f)
            if largest is None or os.path.getsize(candidate) > os.path.getsize(largest):
                largest = candidate
    return largest


def _download_one_file(magnet, file_index, file_name, work_dir, current_idx, total_files):
    input_file = os.path.join(work_dir, "input.txt")
    with open(input_file, "w") as f:
        f.write(magnet + "\n")

    cmd = [
        "aria2c",
        "--enable-dht=true",
        "--enable-dht6=false",
        "--bt-enable-lpd=true",
        "--bt-max-peers=200",
        "--max-connection-per-server=16",
        "--split=16",
        "--min-split-size=1M",
        "--seed-time=0",
        "--file-allocation=none",
        "--summary-interval=1",
        "--console-log-level=notice",
        "--show-console-readout=true",
        "--download-result=full",
        f"--select-file={file_index}",
        f"--input-file={input_file}",
        "--dir", work_dir,
    ]

    last_update = 0
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        line = line.rstrip()
        if not line:
            continue
        parsed = _parse_aria2_line(line)
        if not parsed:
            continue
        now = time.time()
        if now - last_update < 3:
            continue
        last_update = now

        new_state = {
            "phase": "downloading",
            "current_file": file_name,
            "current_index": current_idx,
            "total_files": total_files,
        }
        if "percent" in parsed:
            new_state["percent"] = parsed["percent"]
        if "current_human" in parsed:
            new_state["current_human"] = parsed["current_human"]
        if "total_human" in parsed:
            new_state["total_human"] = parsed["total_human"]
        if "speed_human" in parsed:
            new_state["speed_human"] = parsed["speed_human"]
        if "eta_raw" in parsed:
            new_state["eta_seconds"] = _eta_to_seconds(parsed["eta_raw"])

        _set_state(**new_state)

        backend_payload = {
            "status": "downloading",
            "phase": "download",
            "file_name": file_name,
            "current_file_index": current_idx,
            "total_files": total_files,
        }
        if "percent" in parsed:
            backend_payload["percent"] = parsed["percent"]
        if "current_human" in parsed:
            backend_payload["current_human"] = parsed["current_human"]
        if "total_human" in parsed:
            backend_payload["total_human"] = parsed["total_human"]
        if "speed_human" in parsed:
            backend_payload["speed_human"] = parsed["speed_human"]
        if "eta_raw" in parsed:
            backend_payload["eta_seconds"] = _eta_to_seconds(parsed["eta_raw"])
        _post_update(backend_payload)

    process.wait()
    return process.returncode


def _upload_one_file(local_path, file_name, hf_token, repo_id, repo_type, folder, current_idx, total_files):
    try:
        file_size = os.path.getsize(local_path)
    except Exception:
        file_size = 0

    _set_state(
        phase="uploading",
        current_file=os.path.basename(local_path),
        current_index=current_idx,
        total_files=total_files,
        percent=0,
        current_human="0 B",
        total_human=_format_size(file_size),
        speed_human="0 B/s",
        eta_seconds=None,
        message="Uploading to Hugging Face...",
        error="",
    )

    _post_update({
        "status": "uploading",
        "phase": "upload",
        "save_status": "uploading",
        "file_name": file_name,
        "percent": 0,
        "current_human": "0 B",
        "total_human": _format_size(file_size),
        "speed_human": "0 B/s",
        "current_file_index": current_idx,
        "total_files": total_files,
    })

    stop_flag = {"stop": False}

    def _animate():
        start_ts = time.time()
        last_ts = 0
        while not stop_flag["stop"]:
            time.sleep(0.5)
            if stop_flag["stop"]:
                break

            elapsed = time.time() - start_ts
            if elapsed < 1.5:
                continue

            now = time.time()
            if now - last_ts < 2:
                continue
            last_ts = now

            progress_frac = 1 - pow(0.5, elapsed / 30.0)
            capped = min(progress_frac, 0.95)
            percent_int = int(capped * 100)
            current_bytes = int(file_size * capped)
            speed_bytes = current_bytes / elapsed if elapsed > 0 else 0

            eta_seconds = None
            if speed_bytes > 0 and current_bytes < file_size:
                eta_seconds = int((file_size - current_bytes) / speed_bytes)

            _set_state(
                phase="uploading",
                percent=percent_int,
                current_human=_format_size(current_bytes),
                total_human=_format_size(file_size),
                speed_human=_format_size(speed_bytes) + "/s",
                eta_seconds=eta_seconds,
                message="",
            )

            _post_update({
                "status": "uploading",
                "phase": "upload",
                "save_status": "uploading",
                "file_name": file_name,
                "percent": percent_int,
                "current_human": _format_size(current_bytes),
                "total_human": _format_size(file_size),
                "speed_human": _format_size(speed_bytes) + "/s",
                "eta_seconds": eta_seconds,
                "current_file_index": current_idx,
                "total_files": total_files,
            })

    anim_thread = threading.Thread(target=_animate, daemon=True)
    anim_thread.start()

    try:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        path_in_repo = f"{folder}/{os.path.basename(local_path)}" if folder else os.path.basename(local_path)

        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=path_in_repo,
            repo_id=repo_id,
            repo_type=repo_type,
            commit_message=f"Add {os.path.basename(path_in_repo)}",
        )

        stop_flag["stop"] = True

        prefix = {"dataset": "datasets/", "space": "spaces/"}.get(repo_type, "")
        save_url = f"https://huggingface.co/{prefix}{repo_id}/blob/main/{path_in_repo}"

        try:
            if os.path.exists(local_path):
                os.remove(local_path)
        except Exception:
            pass

        _set_state(
            phase="uploading",
            percent=100,
            current_human=_format_size(file_size),
            total_human=_format_size(file_size),
            speed_human="0 B/s",
            eta_seconds=0,
            message="Uploaded",
        )

        _post_update({
            "status": "saved",
            "save_status": "done",
            "save_url": save_url,
            "file_name": file_name,
            "phase": "upload",
            "percent": 100,
            "current_human": _format_size(file_size),
            "total_human": _format_size(file_size),
            "speed_human": "0 B/s",
            "current_file_index": current_idx,
            "total_files": total_files,
        })

        return True, save_url

    except Exception as e:
        stop_flag["stop"] = True
        err = str(e)[:300]
        _post_update({
            "status": "error",
            "save_status": "error",
            "save_error": err,
            "file_name": file_name,
            "phase": "upload",
            "current_file_index": current_idx,
            "total_files": total_files,
        })
        return False, None


def _run_multi_files(magnet, files, hf_token, repo_id, repo_type, folder):
    if not shutil.which("aria2c"):
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], capture_output=True)

    if not shutil.which("aria2c"):
        _set_state(
            phase="error",
            error="aria2c not available on Colab",
            message="",
        )
        _post_update({
            "status": "error",
            "error": "aria2c not available on Colab",
            "finished": True,
        })
        return

    total_files = len(files)
    completed = 0
    failed = 0

    total_bytes_ok = 0
    total_duration_ok = 0.0

    file_list = []
    for f in files:
        file_list.append({
            "name": f.get("name", ""),
            "size": f.get("size", 0) if isinstance(f, dict) else 0,
            "_status": "pending",
            "_index": f.get("index"),
        })
    _state["files"] = file_list

    _set_state(
        phase="downloading",
        total_files=total_files,
        completed_files=0,
        failed_files=0,
        percent=0,
        current_human="0 B",
        total_human="0 B",
        speed_human="0 B/s",
        eta_seconds=None,
        message=f"Starting download {total_files} file(s)",
        error="",
    )

    downloaded_paths = []

    for idx, f in enumerate(files, start=1):
        file_index = f.get("index")
        file_name = f.get("name") or ""

        if not file_index:
            failed += 1
            if idx - 1 < len(_state["files"]):
                _state["files"][idx - 1]["_status"] = "error"
            _set_state(
                failed_files=failed,
                error=f"Skip: file_index missing for {file_name}",
            )
            continue

        if idx - 1 < len(_state["files"]):
            _state["files"][idx - 1]["_status"] = "downloading"

        _set_state(
            phase="downloading",
            current_file=file_name,
            current_index=idx,
            total_files=total_files,
            completed_files=completed,
            failed_files=failed,
            percent=0,
            current_human="0 B",
            total_human="0 B",
            speed_human="0 B/s",
            eta_seconds=None,
            message="",
            error="",
        )

        work_dir = os.path.join(TEMP_PATH, f"torrent_{int(time.time())}_{idx}")
        os.makedirs(work_dir, exist_ok=True)

        try:
            _post_update({
                "status": "downloading",
                "phase": "download",
                "file_name": file_name,
                "percent": 0,
                "speed_human": "0 B/s",
                "current_file_index": idx,
                "total_files": total_files,
                "completed_files": completed,
                "failed_files": failed,
            })

            rc = _download_one_file(
                magnet=magnet,
                file_index=file_index,
                file_name=file_name,
                work_dir=work_dir,
                current_idx=idx,
                total_files=total_files,
            )

            if rc != 0:
                failed += 1
                err_msg = f"aria2 exit code {rc}"
                if idx - 1 < len(_state["files"]):
                    _state["files"][idx - 1]["_status"] = "error"
                _set_state(
                    failed_files=failed,
                    error=f"[{idx}/{total_files}] Download failed: {err_msg}",
                )
                _post_update({
                    "status": "error",
                    "error": err_msg,
                    "file_name": file_name,
                    "phase": "download",
                    "current_file_index": idx,
                    "total_files": total_files,
                    "completed_files": completed,
                    "failed_files": failed,
                })
                shutil.rmtree(work_dir, ignore_errors=True)
                continue

            actual_file = _find_target_file(work_dir, file_name)
            if not actual_file:
                actual_file = _find_largest_file(work_dir)

            if not actual_file:
                failed += 1
                if idx - 1 < len(_state["files"]):
                    _state["files"][idx - 1]["_status"] = "error"
                _set_state(
                    failed_files=failed,
                    error=f"[{idx}/{total_files}] File not found after download",
                )
                _post_update({
                    "status": "error",
                    "error": "file not found after download",
                    "file_name": file_name,
                    "current_file_index": idx,
                    "total_files": total_files,
                    "completed_files": completed,
                    "failed_files": failed,
                })
                shutil.rmtree(work_dir, ignore_errors=True)
                continue

            if idx - 1 < len(_state["files"]):
                _state["files"][idx - 1]["_status"] = "downloaded"

            _set_state(
                phase="downloading",
                current_file=file_name,
                current_index=idx,
                total_files=total_files,
                completed_files=completed,
                failed_files=failed,
                percent=100,
                current_human=_format_size(os.path.getsize(actual_file)),
                total_human=_format_size(os.path.getsize(actual_file)),
                speed_human="0 B/s",
                eta_seconds=0,
                message="",
                error="",
            )

            _post_update({
                "status": "downloading",
                "phase": "download",
                "file_name": file_name,
                "percent": 100,
                "current_human": _format_size(os.path.getsize(actual_file)),
                "total_human": _format_size(os.path.getsize(actual_file)),
                "speed_human": "0 B/s",
                "current_file_index": idx,
                "total_files": total_files,
                "completed_files": completed,
                "failed_files": failed,
            })

            downloaded_paths.append({
                "idx": idx,
                "file_name": file_name,
                "path": actual_file,
                "work_dir": work_dir,
                "size": os.path.getsize(actual_file),
            })

        except Exception as e:
            failed += 1
            err_msg = str(e)[:200]
            if idx - 1 < len(_state["files"]):
                _state["files"][idx - 1]["_status"] = "error"
            _set_state(
                failed_files=failed,
                error=f"[{idx}/{total_files}] Exception: {err_msg}",
            )
            _post_update({
                "status": "error",
                "error": err_msg,
                "file_name": file_name,
                "phase": "download",
                "current_file_index": idx,
                "total_files": total_files,
                "completed_files": completed,
                "failed_files": failed,
            })
            shutil.rmtree(work_dir, ignore_errors=True)

    _set_state(
        phase="uploading",
        total_files=total_files,
        completed_files=0,
        failed_files=failed,
        message=f"Starting upload {len(downloaded_paths)} file(s)",
        error="",
    )

    for item in downloaded_paths:
        idx = item["idx"]
        file_name = item["file_name"]
        actual_file = item["path"]
        work_dir = item["work_dir"]
        file_size = item["size"]

        if idx - 1 < len(_state["files"]):
            _state["files"][idx - 1]["_status"] = "uploading"

        file_started_at = time.time()

        _set_state(
            phase="uploading",
            current_file=file_name,
            current_index=idx,
            total_files=total_files,
            completed_files=completed,
            failed_files=failed,
            percent=0,
            current_human="0 B",
            total_human=_format_size(file_size),
            speed_human="0 B/s",
            eta_seconds=None,
            message="",
            error="",
        )

        _post_update({
            "status": "uploading",
            "phase": "upload",
            "file_name": file_name,
            "percent": 0,
            "current_human": "0 B",
            "total_human": _format_size(file_size),
            "speed_human": "0 B/s",
            "current_file_index": idx,
            "total_files": total_files,
            "completed_files": completed,
            "failed_files": failed,
        })

        ok, _url = _upload_one_file(
            local_path=actual_file,
            file_name=file_name,
            hf_token=hf_token,
            repo_id=repo_id,
            repo_type=repo_type,
            folder=folder,
            current_idx=idx,
            total_files=total_files,
        )

        file_elapsed = time.time() - file_started_at

        if ok:
            completed += 1
            total_bytes_ok += file_size
            total_duration_ok += file_elapsed
            if idx - 1 < len(_state["files"]):
                _state["files"][idx - 1]["_status"] = "done"
        else:
            failed += 1
            if idx - 1 < len(_state["files"]):
                _state["files"][idx - 1]["_status"] = "error"

        _set_state(
            completed_files=completed,
            failed_files=failed,
        )

        _post_update({
            "current_file_index": idx,
            "total_files": total_files,
            "completed_files": completed,
            "failed_files": failed,
        })

        shutil.rmtree(work_dir, ignore_errors=True)

    final_phase = "done" if failed == 0 else "error"
    final_message = f"Finished: {completed} saved, {failed} failed (from {total_files})"
    final_error = "" if failed == 0 else f"{failed} file(s) failed"

    avg_speed_bps = 0.0
    if total_duration_ok > 0 and total_bytes_ok > 0:
        avg_speed_bps = total_bytes_ok / total_duration_ok

    _set_state(
        phase=final_phase,
        completed_files=completed,
        failed_files=failed,
        percent=100,
        speed_human="0 B/s",
        eta_seconds=None,
        message=final_message,
        error=final_error,
    )

    final_payload = {
        "status": final_phase,
        "finished": True,
        "phase": "done",
        "total_files": total_files,
        "completed_files": completed,
        "failed_files": failed,
        "percent": 100,
        "total_size_bytes": int(total_bytes_ok),
        "total_speed_bps": int(avg_speed_bps),
        "total_elapsed_seconds": float(total_duration_ok),
    }

    if failed > 0:
        final_payload["error"] = f"{failed} file(s) failed out of {total_files}"
        final_payload["save_status"] = "done" if completed > 0 else "error"

    _post_update(final_payload)


def _start_session(session_id, backend_url):
    global SESSION_ID, BACKEND_URL, COLAB_CONFIG, _ui_display

    session_id = (session_id or "").strip()
    backend_url = (backend_url or "").strip()

    if not session_id or not backend_url:
        _set_state(
            phase="error",
            error="Session ID and Backend URL are required",
        )
        return

    SESSION_ID = session_id
    BACKEND_URL = backend_url

    _ui_display = None

    _set_state(
        phase="fetching",
        message=f"Fetching config from {backend_url}",
        error="",
    )

    cfg = _fetch_config(session_id, backend_url)
    if not cfg or not cfg.get("ok"):
        _set_state(
            phase="error",
            error="Failed to fetch config. Check Session ID / Backend URL.",
        )
        return

    COLAB_CONFIG = cfg
    files = cfg.get("files") or []

    if not files:
        _set_state(
            phase="error",
            error="No files in session",
        )
        return

    file_list = []
    for f in files:
        file_list.append({
            "name": f.get("name", ""),
            "size": 0,
            "_status": "pending",
            "_index": f.get("index"),
        })

    _set_state(
        files=file_list,
        total_files=len(files),
        completed_files=0,
        failed_files=0,
        current_file="",
        current_index=0,
        percent=0,
        message=f"Session ready · {len(files)} file(s)",
        error="",
    )

    _run_multi_files(
        magnet=cfg["magnet"],
        files=files,
        hf_token=cfg["token"],
        repo_id=cfg["repo_id"],
        repo_type=cfg["repo_type"],
        folder=cfg.get("folder") or "",
    )


def _on_start_click(session_id, backend_url):
    _start_session(session_id, backend_url)


try:
    from google.colab import output as _colab_output
    _colab_output.register_callback('torrent_start', _on_start_click)
except Exception:
    pass


def _bootstrap():
    _ensure_dirs()

    form_html = """
    <div style="font-family: monospace; margin: 10px 0; padding: 18px;
                background: linear-gradient(135deg, #0f0f1e, #1a1a2e, #16213e);
                border-radius: 12px; border: 1px solid #2a2a4a; color: #e0e0e0;">

        <div style="color: #4CAF50; font-size: 18px; font-weight: bold;
                    letter-spacing: 2px; text-align: center;">
            TORRENT COLAB
        </div>
        <div style="color: #888; font-size: 12px; margin-top: 4px;
                    text-align: center;">
            Download torrent di Colab · Upload otomatis ke Hugging Face
        </div>

        <div style="margin-top: 18px;">
            <label style="font-size: 11px; color: #888;
                          letter-spacing: 1px;">SESSION ID</label>
            <input id="torrent-session-id" type="text"
                   placeholder="paste dari kernel log"
                   style="width: 100%; margin-top: 6px; padding: 10px 12px;
                          background: #0a0a14; border: 1px solid #2a2a4a;
                          border-radius: 6px; color: #e0e0e0;
                          font-family: monospace; font-size: 13px;
                          outline: none; box-sizing: border-box;" />
        </div>

        <div style="margin-top: 12px;">
            <label style="font-size: 11px; color: #888;
                          letter-spacing: 1px;">BACKEND URL</label>
            <input id="torrent-backend-url" type="text"
                   value="https://warthadev-backend.onrender.com"
                   style="width: 100%; margin-top: 6px; padding: 10px 12px;
                          background: #0a0a14; border: 1px solid #2a2a4a;
                          border-radius: 6px; color: #e0e0e0;
                          font-family: monospace; font-size: 13px;
                          outline: none; box-sizing: border-box;" />
        </div>

        <button id="torrent-start-btn"
                style="margin-top: 16px; width: 100%; padding: 12px 20px;
                       background: linear-gradient(90deg, #4CAF50, #00BCD4);
                       border: none; border-radius: 8px; color: white;
                       font-family: monospace; font-size: 14px;
                       font-weight: bold; cursor: pointer;
                       letter-spacing: 1px;">
            START DOWNLOAD
        </button>
    </div>
    """

    js_html = """
    <script>
    (function() {
        var btn = document.getElementById('torrent-start-btn');
        if (!btn) return;
        btn.addEventListener('click', function() {
            var sid = (document.getElementById('torrent-session-id').value || '').trim();
            var backend = (document.getElementById('torrent-backend-url').value || '').trim();
            if (!sid) {
                alert('Isi Session ID dulu');
                return;
            }
            google.colab.kernel.invokeFunction('torrent_start', [sid, backend], {});
        });
    })();
    </script>
    """

    display(HTML(form_html))
    display(HTML(js_html))


_bootstrap()

In [ ]:

#@title Universal Colab Downloader
import os
import re
import time
import json
import shutil
import subprocess
import threading
import requests
import tempfile

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

from IPython.display import display, HTML, Javascript, clear_output

COLAB_SAVE = "/content/Universal"
TEMP_PATH = "/content/Temp"
META_PATH = "/content/meta"

SESSION_ID = None
BACKEND_URL = None
COLAB_CONFIG = None

POST_RETRY_COUNT = 3
POST_RETRY_DELAY = 2

_state = {
    "phase": "idle",
    "files": [],
    "current_file": "",
    "current_index": 0,
    "total_files": 0,
    "completed_files": 0,
    "failed_files": 0,
    "percent": 0,
    "current_human": "0 B",
    "total_human": "0 B",
    "speed_human": "0 B/s",
    "eta_seconds": None,
    "message": "",
    "error": "",
}

_ui_display = None


def _ensure_dirs():
    for p in [COLAB_SAVE, TEMP_PATH, META_PATH]:
        os.makedirs(p, exist_ok=True)


def _format_size(num):
    try:
        n = float(num)
    except Exception:
        return "0 B"
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024.0:
            if unit == "B":
                return f"{int(n)} B"
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} PB"


def _render_ui():
    s = dict(_state)

    phase = s["phase"]
    files = s["files"]
    total_files = s["total_files"]
    completed = s["completed_files"]
    failed = s["failed_files"]
    current_idx = s["current_index"]
    current_file = s["current_file"]
    percent = s["percent"]
    current_human = s["current_human"]
    total_human = s["total_human"]
    speed_human = s["speed_human"]
    eta_seconds = s["eta_seconds"]
    message = s["message"]
    error = s["error"]

    if phase == "idle":
        header_color = "#888"
        header_label = "WAITING FOR INPUT"
    elif phase == "fetching":
        header_color = "#2196F3"
        header_label = "FETCHING CONFIG"
    elif phase == "waiting":
        header_color = "#FF9800"
        header_label = "WAITING FOR USER"
    elif phase == "downloading":
        header_color = "#4CAF50"
        header_label = "DOWNLOADING"
    elif phase == "uploading":
        header_color = "#FF9800"
        header_label = "UPLOADING TO HUGGING FACE"
    elif phase == "done":
        header_color = "#00BCD4"
        header_label = "ALL DONE"
    elif phase == "error":
        header_color = "#F44336"
        header_label = "ERROR"
    else:
        header_color = "#888"
        header_label = phase.upper()

    eta_str = "--"
    if isinstance(eta_seconds, (int, float)) and eta_seconds is not None and eta_seconds >= 0:
        secs = int(eta_seconds)
        if secs < 60:
            eta_str = f"{secs}s"
        elif secs < 3600:
            eta_str = f"{secs / 60:.1f}m"
        else:
            eta_str = f"{secs / 3600:.1f}h"

    if total_files > 0:
        overall_percent = int(((completed + failed) / total_files) * 100)
    else:
        overall_percent = 0

    rows_html = ""
    if files:
        for idx, f in enumerate(files, start=1):
            name = f.get("name", "")
            size = f.get("size", 0)
            status = f.get("_status", "pending")

            if status == "done":
                badge_bg = "#4CAF50"
                badge_text = "DONE"
                row_color = "#4CAF50"
            elif status == "uploading":
                badge_bg = "#FF9800"
                badge_text = "UPLOAD"
                row_color = "#FF9800"
            elif status == "downloading":
                badge_bg = "#2196F3"
                badge_text = "DOWNLOAD"
                row_color = "#2196F3"
            elif status == "downloaded":
                badge_bg = "#00BCD4"
                badge_text = "READY"
                row_color = "#00BCD4"
            elif status == "error":
                badge_bg = "#F44336"
                badge_text = "FAILED"
                row_color = "#F44336"
            else:
                badge_bg = "#555"
                badge_text = "PENDING"
                row_color = "#888"

            size_str = _format_size(size) if size else ""

            rows_html += f"""
            <div style="display: grid; grid-template-columns: 34px 1fr 90px 80px;
                        padding: 8px 10px; border-bottom: 1px solid #2a2a2a;
                        color: {row_color}; font-family: monospace; font-size: 12px;
                        align-items: center; gap: 8px;">
                <span style="color: #888; font-size: 11px;">#{idx}</span>
                <span style="overflow: hidden; text-overflow: ellipsis;
                             white-space: nowrap;">{name}</span>
                <span style="text-align: right; color: #aaa; font-size: 11px;">{size_str}</span>
                <span style="text-align: center; background: {badge_bg}; color: #fff;
                             border-radius: 4px; font-size: 10px; padding: 2px 6px;
                             font-weight: bold;">{badge_text}</span>
            </div>
            """
    else:
        rows_html = """
        <div style="padding: 16px; text-align: center; color: #666;
                    font-family: monospace; font-size: 12px;">
            No files loaded yet
        </div>
        """

    error_html = ""
    if error:
        error_html = f"""
        <div style="margin-top: 10px; padding: 10px 14px; background: #2a1a1a;
                    border-left: 4px solid #F44336; border-radius: 6px;
                    color: #F44336; font-family: monospace; font-size: 12px;
                    word-break: break-all;">
            {error}
        </div>
        """

    message_html = ""
    if message:
        message_html = f"""
        <div style="margin-top: 10px; padding: 10px 14px; background: #1a1a2e;
                    border-left: 4px solid {header_color}; border-radius: 6px;
                    color: #e0e0e0; font-family: monospace; font-size: 12px;">
            {message}
        </div>
        """

    progress_html = ""
    if phase in ("downloading", "uploading"):
        progress_html = f"""
        <div style="margin-top: 12px; padding: 12px 14px; background: #141420;
                    border-radius: 8px; border: 1px solid #2a2a4a;">
            <div style="display: flex; justify-content: space-between;
                        align-items: center; margin-bottom: 8px;">
                <span style="color: #e0e0e0; font-family: monospace; font-size: 12px;
                             overflow: hidden; text-overflow: ellipsis;
                             white-space: nowrap; max-width: 60%;">
                    {current_file}
                </span>
                <span style="color: {header_color}; font-family: monospace;
                             font-size: 11px; font-weight: bold;">
                    [{current_idx}/{total_files}] {header_label}
                </span>
            </div>
            <div style="background: #2a2a2a; border-radius: 6px; height: 18px;
                        overflow: hidden; position: relative;">
                <div style="background: linear-gradient(90deg, {header_color}, {header_color}cc);
                            height: 100%; width: {percent}%;
                            transition: width 0.3s linear; border-radius: 6px;
                            display: flex; align-items: center; justify-content: center;
                            color: #fff; font-size: 10px; font-weight: bold;">
                    {percent}%
                </div>
            </div>
            <div style="display: flex; justify-content: space-between;
                        margin-top: 6px; color: #888; font-family: monospace;
                        font-size: 11px;">
                <span>{current_human} / {total_human}</span>
                <span>{speed_human} · ETA {eta_str}</span>
            </div>
        </div>
        """

    html = f"""
    <div style="font-family: monospace; margin: 10px 0;
                background: linear-gradient(135deg, #0f0f1e, #1a1a2e, #16213e);
                border-radius: 12px; border: 1px solid #2a2a4a; padding: 16px;
                color: #e0e0e0;">
        <div style="display: flex; justify-content: space-between;
                    align-items: center; margin-bottom: 12px;">
            <div style="color: #4CAF50; font-size: 16px; font-weight: bold;
                        letter-spacing: 1px;">
                UNIVERSAL COLAB
            </div>
            <div style="color: {header_color}; font-size: 11px; font-weight: bold;
                        background: rgba(255,255,255,0.05); padding: 4px 10px;
                        border-radius: 20px; border: 1px solid {header_color};">
                {header_label}
            </div>
        </div>

        <div style="display: flex; gap: 16px; margin-bottom: 12px;
                    font-size: 11px; color: #888;">
            <span>Total: <b style="color: #e0e0e0;">{total_files}</b></span>
            <span>Done: <b style="color: #4CAF50;">{completed}</b></span>
            <span>Failed: <b style="color: #F44336;">{failed}</b></span>
            <span>Overall: <b style="color: #00BCD4;">{overall_percent}%</b></span>
        </div>

        <div style="background: #141420; border-radius: 8px; border: 1px solid #2a2a4a;
                    max-height: 320px; overflow-y: auto;">
            {rows_html}
        </div>

        {progress_html}
        {message_html}
        {error_html}
    </div>
    """
    return html


def _ui_update():
    global _ui_display
    html = _render_ui()
    if _ui_display is None:
        _ui_display = display(HTML(html), display_id=True)
    else:
        try:
            _ui_display.update(HTML(html))
        except Exception:
            _ui_display = display(HTML(html), display_id=True)


def _set_state(**kwargs):
    _state.update(kwargs)
    try:
        _ui_update()
    except Exception:
        pass


def _post_update(payload):
    if not SESSION_ID or not BACKEND_URL:
        return False
    url = f"{BACKEND_URL.rstrip('/')}/universal/colab-update"
    params = {"session_id": SESSION_ID}

    for attempt in range(POST_RETRY_COUNT):
        try:
            r = requests.post(url, params=params, json=payload, timeout=15)
            if r.ok:
                return True
        except Exception:
            pass
        if attempt < POST_RETRY_COUNT - 1:
            time.sleep(POST_RETRY_DELAY * (attempt + 1))
    return False


def _fetch_config(session_id, backend_url):
    url = f"{backend_url.rstrip('/')}/universal/colab-config"
    params = {"session_id": session_id}
    for attempt in range(POST_RETRY_COUNT):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.ok:
                return r.json()
        except Exception:
            pass
        if attempt < POST_RETRY_COUNT - 1:
            time.sleep(POST_RETRY_DELAY * (attempt + 1))
    return None


def _check_status(session_id, backend_url):
    url = f"{backend_url.rstrip('/')}/universal/colab-check"
    params = {"session_id": session_id}
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.ok:
            return r.json().get("status")
    except Exception:
        pass
    return None


def _create_temp_dir():
    try:
        return tempfile.mkdtemp(prefix="uvd_", dir=TEMP_PATH)
    except Exception:
        return tempfile.mkdtemp(prefix="uvd_", dir="/tmp")


def _cleanup(temp_dir):
    if not temp_dir:
        return
    try:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, ignore_errors=True)
    except Exception:
        pass


def _move_final(src, dst):
    if not src or not os.path.exists(src):
        return False, "src missing"
    try:
        dst_dir = os.path.dirname(dst) or "."
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(dst):
            try:
                os.remove(dst)
            except Exception:
                pass
        shutil.move(src, dst)
        return True, None
    except Exception:
        try:
            shutil.copy2(src, dst)
            os.remove(src)
            return True, None
        except Exception as e2:
            return False, f"move fail: {str(e2)[:200]}"


def _safe_filename(name, ext):
    base = re.sub(r'[<>:"/\\|?*]', "", name or "download")
    base = re.sub(r"\s+", " ", base).strip()
    if not base:
        base = "download"
    if len(base) > 120:
        base = base[:120]
    if ext and not base.lower().endswith("." + ext.lower()):
        base = f"{base}.{ext}"
    return base


def _ytdlp_probe(url):
    from yt_dlp import YoutubeDL
    strategies = [
        {"client": None}, {"client": ["android", "web_safari"]},
        {"client": ["ios", "mweb"]}, {"client": ["tv_embedded", "web_embedded"]},
        {"client": ["web"]},
    ]
    for s in strategies:
        opts = {"quiet": True, "no_warnings": True, "socket_timeout": 60, "retries": 5, "noplaylist": True}
        if s["client"]:
            opts["extractor_args"] = {"youtube": {"player_client": s["client"]}}
        try:
            with YoutubeDL(opts) as ydl:
                info = ydl.extract_info(url, download=False)
                if info and (info.get("formats") or info.get("url")):
                    return info
        except Exception:
            continue
    return None


def _download_with_ytdlp(url, output_path, format_id, file_name, current_idx, total_files):
    from yt_dlp import YoutubeDL
    temp_dir = _create_temp_dir()
    target_basename = os.path.basename(output_path)
    temp_output = os.path.join(temp_dir, target_basename)

    hook_state = {'start_time': time.time()}

    def _hook(d):
        if d['status'] == 'downloading':
            if hook_state['start_time'] == 0:
                hook_state['start_time'] = time.time()

            pct_str = d.get('_percent_str', '0%').replace('%','')
            try: pct_val = float(pct_str)
            except: pct_val = 0

            spd_str = d.get('_speed_str', '0 B/s')
            cur_str = d.get('_downloaded_bytes_str', '0 B')
            tot_str = d.get('_total_bytes_str', '0 B')
            eta = d.get('eta')

            _set_state(
                phase="downloading",
                current_file=file_name,
                current_index=current_idx,
                total_files=total_files,
                percent=pct_val,
                current_human=cur_str,
                total_human=tot_str,
                speed_human=spd_str,
                eta_seconds=eta
            )

            _post_update({
                "status": "downloading",
                "phase": "download",
                "file_name": file_name,
                "percent": pct_val,
                "current_human": cur_str,
                "total_human": tot_str,
                "speed_human": spd_str,
                "eta_seconds": eta
            })

    opts = {
        "outtmpl": temp_output, "format": format_id, "merge_output_format": "mp4",
        "quiet": True, "no_warnings": True, "noprogress": True,
        "fragment_retries": 30, "retries": 30, "socket_timeout": 60, "concurrent_fragment_downloads": 32,
        "http_chunk_size": 20 * 1024 * 1024, "ffmpeg_location": shutil.which("ffmpeg"),
        "paths": {"temp": temp_dir}, "noplaylist": True,
        "progress_hooks": [_hook],
    }

    try:
        with YoutubeDL(opts) as ydl:
            ydl.download([url])

        actual = temp_output
        if not os.path.exists(actual):
            base_noext, _ = os.path.splitext(temp_output)
            for ext in [".mp4", ".mkv", ".webm", ".ts", ".flv", ".m4a", ".mp3", ".opus"]:
                if os.path.exists(base_noext + ext):
                    actual = base_noext + ext
                    break
        if not os.path.exists(actual):
            return False, "yt-dlp no output"

        moved, err = _move_final(actual, output_path)
        return moved, err
    except Exception as e:
        return False, str(e)[:200]
    finally:
        _cleanup(temp_dir)


def _upload_one_file(local_path, file_name, hf_token, repo_id, repo_type, folder, current_idx, total_files):
    try:
        file_size = os.path.getsize(local_path)
    except Exception:
        file_size = 0

    _set_state(
        phase="uploading",
        current_file=os.path.basename(local_path),
        current_index=current_idx,
        total_files=total_files,
        percent=0,
        current_human="0 B",
        total_human=_format_size(file_size),
        speed_human="0 B/s",
        eta_seconds=None,
        message="Uploading to Hugging Face...",
        error="",
    )

    _post_update({
        "status": "uploading",
        "phase": "upload",
        "save_status": "uploading",
        "file_name": file_name,
        "percent": 0,
        "current_human": "0 B",
        "total_human": _format_size(file_size),
        "speed_human": "0 B/s",
        "current_file_index": current_idx,
        "total_files": total_files,
    })

    stop_flag = {"stop": False}

    def _animate():
        start_ts = time.time()
        last_ts = 0
        while not stop_flag["stop"]:
            time.sleep(0.5)
            if stop_flag["stop"]:
                break

            elapsed = time.time() - start_ts
            if elapsed < 1.5:
                continue

            now = time.time()
            if now - last_ts < 2:
                continue
            last_ts = now

            progress_frac = 1 - pow(0.5, elapsed / 30.0)
            capped = min(progress_frac, 0.95)
            percent_int = int(capped * 100)
            current_bytes = int(file_size * capped)
            speed_bytes = current_bytes / elapsed if elapsed > 0 else 0

            eta_seconds = None
            if speed_bytes > 0 and current_bytes < file_size:
                eta_seconds = int((file_size - current_bytes) / speed_bytes)

            _set_state(
                phase="uploading",
                percent=percent_int,
                current_human=_format_size(current_bytes),
                total_human=_format_size(file_size),
                speed_human=_format_size(speed_bytes) + "/s",
                eta_seconds=eta_seconds,
                message="",
            )

            _post_update({
                "status": "uploading",
                "phase": "upload",
                "save_status": "uploading",
                "file_name": file_name,
                "percent": percent_int,
                "current_human": _format_size(current_bytes),
                "total_human": _format_size(file_size),
                "speed_human": _format_size(speed_bytes) + "/s",
                "eta_seconds": eta_seconds,
                "current_file_index": current_idx,
                "total_files": total_files,
            })

    anim_thread = threading.Thread(target=_animate, daemon=True)
    anim_thread.start()

    try:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        path_in_repo = f"{folder}/{os.path.basename(local_path)}" if folder else os.path.basename(local_path)

        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=path_in_repo,
            repo_id=repo_id,
            repo_type=repo_type,
            commit_message=f"Add {os.path.basename(path_in_repo)}",
        )

        stop_flag["stop"] = True

        prefix = {"dataset": "datasets/", "space": "spaces/", "model": "models/"}.get(repo_type, "")
        save_url = f"https://huggingface.co/{prefix}{repo_id}/blob/main/{path_in_repo}"

        try:
            if os.path.exists(local_path):
                os.remove(local_path)
        except Exception:
            pass

        _set_state(
            phase="uploading",
            percent=100,
            current_human=_format_size(file_size),
            total_human=_format_size(file_size),
            speed_human="0 B/s",
            eta_seconds=0,
            message="Uploaded",
        )

        _post_update({
            "status": "saved",
            "save_status": "done",
            "save_url": save_url,
            "file_name": file_name,
            "phase": "upload",
            "percent": 100,
            "current_human": _format_size(file_size),
            "total_human": _format_size(file_size),
            "speed_human": "0 B/s",
            "current_file_index": current_idx,
            "total_files": total_files,
        })

        return True, save_url
    except Exception as e:
        stop_flag["stop"] = True
        err = str(e)[:300]
        _post_update({
            "status": "error",
            "save_status": "error",
            "save_error": err,
            "file_name": file_name,
            "phase": "upload",
            "current_file_index": current_idx,
            "total_files": total_files,
        })
        return False, None


def _start_session(session_id, backend_url):
    global SESSION_ID, BACKEND_URL, COLAB_CONFIG, _ui_display

    session_id = (session_id or "").strip()
    backend_url = (backend_url or "").strip()

    if not session_id or not backend_url:
        _set_state(phase="error", error="Session ID and Backend URL are required")
        return

    SESSION_ID = session_id
    BACKEND_URL = backend_url
    _ui_display = None

    _set_state(phase="fetching", message=f"Fetching config from {backend_url}", error="")

    cfg = _fetch_config(session_id, backend_url)
    if not cfg or not cfg.get("ok"):
        _set_state(phase="error", error="Failed to fetch config. Check Session ID / Backend URL.")
        return

    COLAB_CONFIG = cfg
    url = cfg.get("url")

    _set_state(phase="waiting", message="Analyzing source URL...")
    info = _ytdlp_probe(url)

    if not info:
        _set_state(phase="error", error="Failed to extract. Blocked or not supported.")
        _post_update({"status": "error", "phase": "resolve", "error": "Failed to extract.", "finished": True})
        return

    title = info.get("title") or "download"
    formats = []
    idx = 1
    for f in info.get("formats", []):
        height = f.get("height")
        vcodec = (f.get("vcodec") or "none").lower()
        acodec = (f.get("acodec") or "none").lower()
        if vcodec == "none" and acodec == "none": continue
        if height and vcodec != "none":
            formats.append({"index": idx, "name": f"{title} [{height}p] ({f.get('ext', 'mp4')})", "size": f.get("filesize") or f.get("filesize_approx") or 0, "format_id": f.get("format_id")})
            idx += 1
        elif acodec != "none" and vcodec == "none":
            formats.append({"index": idx, "name": f"{title} [Audio] ({f.get('ext', 'm4a')})", "size": f.get("filesize") or f.get("filesize_approx") or 0, "format_id": f.get("format_id")})
            idx += 1

    if not formats:
        formats.append({"index": 1, "name": f"{title} [Best Auto]", "size": 0, "format_id": "best"})

    file_list = []
    for f in formats:
        file_list.append({
            "name": f.get("name", ""),
            "size": f.get("size", 0),
            "_status": "pending",
            "_index": f.get("index"),
        })

    _state["files"] = file_list
    _set_state(
        phase="waiting",
        total_files=len(formats),
        message="Pilih format di web, Colab akan otomatis lanjut.",
    )

    _post_update({"status": "waiting", "phase": "resolve", "finished": False, "formats": formats})

    while True:
        time.sleep(3)
        status = _check_status(SESSION_ID, BACKEND_URL)
        if status == "download":
            break
        elif status in ["cancel", "error"]:
            _set_state(phase="error", error="Dibatalkan oleh user.")
            return

    cfg = _fetch_config(SESSION_ID, BACKEND_URL)
    if not cfg or not cfg.get("formats"):
        _set_state(phase="error", error="Gagal mengambil daftar pilihan dari web.")
        return

    selected_formats = cfg.get("formats")
    hf_token = cfg.get("token")
    repo_id = cfg.get("repo_id")
    repo_type = cfg.get("repo_type") or "dataset"
    folder = cfg.get("folder") or ""

    total_files = len(selected_formats)
    completed = 0
    failed = 0

    _set_state(
        phase="downloading",
        total_files=total_files,
        completed_files=0,
        failed_files=0,
        message=f"Mulai download {total_files} file.",
        error="",
    )

    downloaded_paths = []

    for idx, item in enumerate(selected_formats, start=1):
        fname = _safe_filename(item.get("name", "download"), "")
        ext = fname.split('.')[-1] if '.' in fname else 'mp4'

        _state["files"][idx - 1]["_status"] = "downloading"
        _set_state(current_file=fname, current_index=idx)

        temp_out = tempfile.mktemp(suffix=f".{ext}", prefix="uvd_video_", dir=TEMP_PATH)
        ok, err = _download_with_ytdlp(url, temp_out, format_id=item.get("format_id"), file_name=fname, current_idx=idx, total_files=total_files)

        if ok:
            _state["files"][idx - 1]["_status"] = "downloaded"
            downloaded_paths.append({
                "idx": idx,
                "file_name": fname,
                "path": temp_out,
                "size": os.path.getsize(temp_out),
            })
        else:
            failed += 1
            _state["files"][idx - 1]["_status"] = "error"
            _set_state(failed_files=failed, error=f"[{idx}/{total_files}] Gagal: {err}")
            _post_update({"status": "error", "phase": "download", "error": err, "finished": True})
            return

    _set_state(phase="uploading", completed_files=0, message=f"Mulai upload {len(downloaded_paths)} file.")

    for item in downloaded_paths:
        idx = item["idx"]
        fname = item["file_name"]
        fpath = item["path"]

        _state["files"][idx - 1]["_status"] = "uploading"
        _set_state(current_file=fname, current_index=idx)

        ok, _ = _upload_one_file(fpath, fname, hf_token, repo_id, repo_type, folder, idx, total_files)

        if ok:
            completed += 1
            _state["files"][idx - 1]["_status"] = "done"
        else:
            failed += 1
            _state["files"][idx - 1]["_status"] = "error"

        _set_state(completed_files=completed, failed_files=failed)

    final_phase = "done" if failed == 0 else "error"
    _set_state(phase=final_phase, percent=100, message=f"Selesai: {completed} OK, {failed} Gagal.")
    _post_update({"status": "done" if failed == 0 else "error", "phase": "done", "finished": True, "percent": 100})


def _on_start_click(session_id, backend_url):
    _start_session(session_id, backend_url)

try:
    from google.colab import output as _colab_output
    _colab_output.register_callback('universal_start', _on_start_click)
except Exception:
    pass


def _bootstrap():
    _ensure_dirs()

    form_html = """
    <div style="font-family: monospace; margin: 10px 0; padding: 18px;
                background: linear-gradient(135deg, #0f0f1e, #1a1a2e, #16213e);
                border-radius: 12px; border: 1px solid #2a2a4a; color: #e0e0e0;">

        <div style="color: #4CAF50; font-size: 18px; font-weight: bold;
                    letter-spacing: 2px; text-align: center;">
            UNIVERSAL COLAB
        </div>
        <div style="color: #888; font-size: 12px; margin-top: 4px;
                    text-align: center;">
            Download media universal di Colab · Upload otomatis ke Hugging Face
        </div>

        <div style="margin-top: 18px;">
            <label style="font-size: 11px; color: #888;
                          letter-spacing: 1px;">SESSION ID</label>
            <input id="uvd-session-id" type="text"
                   placeholder="paste dari kernel log"
                   style="width: 100%; margin-top: 6px; padding: 10px 12px;
                          background: #0a0a14; border: 1px solid #2a2a4a;
                          border-radius: 6px; color: #e0e0e0;
                          font-family: monospace; font-size: 13px;
                          outline: none; box-sizing: border-box;" />
        </div>

        <div style="margin-top: 12px;">
            <label style="font-size: 11px; color: #888;
                          letter-spacing: 1px;">BACKEND URL</label>
            <input id="uvd-backend-url" type="text"
                   value="https://warthadev-backend.onrender.com"
                   style="width: 100%; margin-top: 6px; padding: 10px 12px;
                          background: #0a0a14; border: 1px solid #2a2a4a;
                          border-radius: 6px; color: #e0e0e0;
                          font-family: monospace; font-size: 13px;
                          outline: none; box-sizing: border-box;" />
        </div>

        <button id="uvd-start-btn"
                style="margin-top: 16px; width: 100%; padding: 12px 20px;
                       background: linear-gradient(90deg, #4CAF50, #00BCD4);
                       border: none; border-radius: 8px; color: white;
                       font-family: monospace; font-size: 14px;
                       font-weight: bold; cursor: pointer;
                       letter-spacing: 1px;">
            START DOWNLOAD
        </button>
    </div>
    """

    js_html = """
    <script>
    (function() {
        var btn = document.getElementById('uvd-start-btn');
        if (!btn) return;
        btn.addEventListener('click', function() {
            var sid = (document.getElementById('uvd-session-id').value || '').trim();
            var backend = (document.getElementById('uvd-backend-url').value || '').trim();
            if (!sid) {
                alert('Isi Session ID dulu');
                return;
            }
            google.colab.kernel.invokeFunction('universal_start', [sid, backend], {});
        });
    })();
    </script>
    """

    display(HTML(form_html))
    display(HTML(js_html))


_bootstrap()

In [ ]:

#@title Universal Colab Downloader v6.11 (Direct Fix + aria2c)
import os
import re
import time
import json
import shutil
import subprocess
import threading
import requests
import tempfile

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

from IPython.display import display, HTML, Javascript, clear_output

COLAB_SAVE = "/content/Universal"
TEMP_PATH = "/content/Temp"
META_PATH = "/content/meta"

SESSION_ID = None
BACKEND_URL = None
COLAB_CONFIG = None

POST_RETRY_COUNT = 3
POST_RETRY_DELAY = 2

DIRECT_EXTS = (".mp4", ".mkv", ".webm", ".avi", ".ts", ".mp3", ".m4a", ".zip", ".rar", ".7z", ".pdf")
UA_DEFAULT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

_state = {
    "phase": "idle",
    "files": [],
    "current_file": "",
    "current_index": 0,
    "total_files": 0,
    "completed_files": 0,
    "failed_files": 0,
    "percent": 0,
    "current_human": "0 B",
    "total_human": "0 B",
    "speed_human": "0 B/s",
    "eta_seconds": None,
    "message": "",
    "error": "",
}

_ui_display = None

def _ensure_dirs():
    for p in [COLAB_SAVE, TEMP_PATH, META_PATH]:
        os.makedirs(p, exist_ok=True)

def _format_size(num):
    try:
        n = float(num)
    except Exception:
        return "0 B"
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024.0:
            if unit == "B":
                return f"{int(n)} B"
            return f"{n:.1f} {unit}"
        n /= 1024.0
    return f"{n:.1f} PB"

def _render_ui():
    s = dict(_state)

    phase = s["phase"]
    files = s["files"]
    total_files = s["total_files"]
    completed = s["completed_files"]
    failed = s["failed_files"]
    current_idx = s["current_index"]
    current_file = s["current_file"]
    percent = s["percent"]
    current_human = s["current_human"]
    total_human = s["total_human"]
    speed_human = s["speed_human"]
    eta_seconds = s["eta_seconds"]
    message = s["message"]
    error = s["error"]

    if phase == "idle":
        header_color = "#888"
        header_label = "WAITING FOR INPUT"
    elif phase == "fetching":
        header_color = "#2196F3"
        header_label = "FETCHING CONFIG"
    elif phase == "waiting":
        header_color = "#FF9800"
        header_label = "WAITING FOR USER"
    elif phase == "downloading":
        header_color = "#4CAF50"
        header_label = "DOWNLOADING"
    elif phase == "uploading":
        header_color = "#FF9800"
        header_label = "UPLOADING TO HUGGING FACE"
    elif phase == "done":
        header_color = "#00BCD4"
        header_label = "ALL DONE"
    elif phase == "error":
        header_color = "#F44336"
        header_label = "ERROR"
    else:
        header_color = "#888"
        header_label = phase.upper()

    eta_str = "--"
    if isinstance(eta_seconds, (int, float)) and eta_seconds is not None and eta_seconds >= 0:
        secs = int(eta_seconds)
        if secs < 60:
            eta_str = f"{secs}s"
        elif secs < 3600:
            eta_str = f"{secs / 60:.1f}m"
        else:
            eta_str = f"{secs / 3600:.1f}h"

    if total_files > 0:
        overall_percent = int(((completed + failed) / total_files) * 100)
    else:
        overall_percent = 0

    rows_html = ""
    if files:
        for idx, f in enumerate(files, start=1):
            name = f.get("name", "")
            size = f.get("size", 0)
            status = f.get("_status", "pending")

            if status == "done":
                badge_bg = "#4CAF50"
                badge_text = "DONE"
                row_color = "#4CAF50"
            elif status == "uploading":
                badge_bg = "#FF9800"
                badge_text = "UPLOAD"
                row_color = "#FF9800"
            elif status == "downloading":
                badge_bg = "#2196F3"
                badge_text = "DOWNLOAD"
                row_color = "#2196F3"
            elif status == "downloaded":
                badge_bg = "#00BCD4"
                badge_text = "READY"
                row_color = "#00BCD4"
            elif status == "error":
                badge_bg = "#F44336"
                badge_text = "FAILED"
                row_color = "#F44336"
            else:
                badge_bg = "#555"
                badge_text = "PENDING"
                row_color = "#888"

            size_str = _format_size(size) if size else ""

            rows_html += f"""
            <div style="display: grid; grid-template-columns: 34px 1fr 90px 80px;
                        padding: 8px 10px; border-bottom: 1px solid #2a2a2a;
                        color: {row_color}; font-family: monospace; font-size: 12px;
                        align-items: center; gap: 8px;">
                <span style="color: #888; font-size: 11px;">#{idx}</span>
                <span style="overflow: hidden; text-overflow: ellipsis;
                             white-space: nowrap;">{name}</span>
                <span style="text-align: right; color: #aaa; font-size: 11px;">{size_str}</span>
                <span style="text-align: center; background: {badge_bg}; color: #fff;
                             border-radius: 4px; font-size: 10px; padding: 2px 6px;
                             font-weight: bold;">{badge_text}</span>
            </div>
            """
    else:
        rows_html = """
        <div style="padding: 16px; text-align: center; color: #666;
                    font-family: monospace; font-size: 12px;">
            No files loaded yet
        </div>
        """

    error_html = ""
    if error:
        error_html = f"""
        <div style="margin-top: 10px; padding: 10px 14px; background: #2a1a1a;
                    border-left: 4px solid #F44336; border-radius: 6px;
                    color: #F44336; font-family: monospace; font-size: 12px;
                    word-break: break-all;">
            {error}
        </div>
        """

    message_html = ""
    if message:
        message_html = f"""
        <div style="margin-top: 10px; padding: 10px 14px; background: #1a1a2e;
                    border-left: 4px solid {header_color}; border-radius: 6px;
                    color: #e0e0e0; font-family: monospace; font-size: 12px;">
            {message}
        </div>
        """

    progress_html = ""
    if phase in ("downloading", "uploading"):
        progress_html = f"""
        <div style="margin-top: 12px; padding: 12px 14px; background: #141420;
                    border-radius: 8px; border: 1px solid #2a2a4a;">
            <div style="display: flex; justify-content: space-between;
                        align-items: center; margin-bottom: 8px;">
                <span style="color: #e0e0e0; font-family: monospace; font-size: 12px;
                             overflow: hidden; text-overflow: ellipsis;
                             white-space: nowrap; max-width: 60%;">
                    {current_file}
                </span>
                <span style="color: {header_color}; font-family: monospace;
                             font-size: 11px; font-weight: bold;">
                    [{current_idx}/{total_files}] {header_label}
                </span>
            </div>
            <div style="background: #2a2a2a; border-radius: 6px; height: 18px;
                        overflow: hidden; position: relative;">
                <div style="background: linear-gradient(90deg, {header_color}, {header_color}cc);
                            height: 100%; width: {percent}%;
                            transition: width 0.3s linear; border-radius: 6px;
                            display: flex; align-items: center; justify-content: center;
                            color: #fff; font-size: 10px; font-weight: bold;">
                    {percent}%
                </div>
            </div>
            <div style="display: flex; justify-content: space-between;
                        margin-top: 6px; color: #888; font-family: monospace;
                        font-size: 11px;">
                <span>{current_human} / {total_human}</span>
                <span>{speed_human} · ETA {eta_str}</span>
            </div>
        </div>
        """

    html = f"""
    <div style="font-family: monospace; margin: 10px 0;
                background: linear-gradient(135deg, #0f0f1e, #1a1a2e, #16213e);
                border-radius: 12px; border: 1px solid #2a2a4a; padding: 16px;
                color: #e0e0e0;">
        <div style="display: flex; justify-content: space-between;
                    align-items: center; margin-bottom: 12px;">
            <div style="color: #4CAF50; font-size: 16px; font-weight: bold;
                        letter-spacing: 1px;">
                UNIVERSAL COLAB
            </div>
            <div style="color: {header_color}; font-size: 11px; font-weight: bold;
                        background: rgba(255,255,255,0.05); padding: 4px 10px;
                        border-radius: 20px; border: 1px solid {header_color};">
                {header_label}
            </div>
        </div>

        <div style="display: flex; gap: 16px; margin-bottom: 12px;
                    font-size: 11px; color: #888;">
            <span>Total: <b style="color: #e0e0e0;">{total_files}</b></span>
            <span>Done: <b style="color: #4CAF50;">{completed}</b></span>
            <span>Failed: <b style="color: #F44336;">{failed}</b></span>
            <span>Overall: <b style="color: #00BCD4;">{overall_percent}%</b></span>
        </div>

        <div style="background: #141420; border-radius: 8px; border: 1px solid #2a2a4a;
                    max-height: 320px; overflow-y: auto;">
            {rows_html}
        </div>

        {progress_html}
        {message_html}
        {error_html}
    </div>
    """
    return html

def _ui_update():
    global _ui_display
    html = _render_ui()
    if _ui_display is None:
        _ui_display = display(HTML(html), display_id=True)
    else:
        try:
            _ui_display.update(HTML(html))
        except Exception:
            _ui_display = display(HTML(html), display_id=True)

def _set_state(**kwargs):
    _state.update(kwargs)
    try:
        _ui_update()
    except Exception:
        pass

def _post_update(payload):
    if not SESSION_ID or not BACKEND_URL:
        return False
    url = f"{BACKEND_URL.rstrip('/')}/universal/colab-update"
    params = {"session_id": SESSION_ID}
    for attempt in range(POST_RETRY_COUNT):
        try:
            r = requests.post(url, params=params, json=payload, timeout=15)
            if r.ok:
                return True
        except Exception:
            pass
        if attempt < POST_RETRY_COUNT - 1:
            time.sleep(POST_RETRY_DELAY * (attempt + 1))
    return False

def _fetch_config(session_id, backend_url):
    url = f"{backend_url.rstrip('/')}/universal/colab-config"
    params = {"session_id": session_id}
    for attempt in range(POST_RETRY_COUNT):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.ok:
                return r.json()
        except Exception:
            pass
        if attempt < POST_RETRY_COUNT - 1:
            time.sleep(POST_RETRY_DELAY * (attempt + 1))
    return None

def _check_status(session_id, backend_url):
    url = f"{backend_url.rstrip('/')}/universal/colab-check"
    params = {"session_id": session_id}
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.ok:
            return r.json().get("status")
    except Exception:
        pass
    return None

def _create_temp_dir():
    try:
        return tempfile.mkdtemp(prefix="uvd_", dir=TEMP_PATH)
    except Exception:
        return tempfile.mkdtemp(prefix="uvd_", dir="/tmp")

def _cleanup(temp_dir):
    if not temp_dir:
        return
    try:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, ignore_errors=True)
    except Exception:
        pass

def _move_final(src, dst):
    if not src or not os.path.exists(src):
        return False, "src missing"
    try:
        dst_dir = os.path.dirname(dst) or "."
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(dst):
            try:
                os.remove(dst)
            except Exception:
                pass
        shutil.move(src, dst)
        return True, None
    except Exception:
        try:
            shutil.copy2(src, dst)
            os.remove(src)
            return True, None
        except Exception as e2:
            return False, f"move fail: {str(e2)[:200]}"

def _safe_filename(name, ext):
    base = re.sub(r'[<>:"/\\|?*]', "", name or "download")
    base = re.sub(r"\s+", " ", base).strip()
    if not base:
        base = "download"
    if len(base) > 120:
        base = base[:120]
    if ext and not base.lower().endswith("." + ext.lower()):
        base = f"{base}.{ext}"
    return base

def _ytdlp_probe(url):
    from yt_dlp import YoutubeDL
    strategies = [
        {"client": None}, {"client": ["android", "web_safari"]},
        {"client": ["ios", "mweb"]}, {"client": ["tv_embedded", "web_embedded"]},
        {"client": ["web"]},
    ]
    for s in strategies:
        opts = {"quiet": True, "no_warnings": True, "socket_timeout": 60, "retries": 5, "noplaylist": True}
        if s["client"]:
            opts["extractor_args"] = {"youtube": {"player_client": s["client"]}}
        try:
            with YoutubeDL(opts) as ydl:
                info = ydl.extract_info(url, download=False)
                if info and (info.get("formats") or info.get("url")):
                    return info
        except Exception:
            continue
    return None

def _parse_aria2_line(line):
    out = {}
    m = re.search(r"\((\d+)%\)", line)
    if m: out["percent"] = int(m.group(1))
    m2 = re.search(r"([\d.]+\s*[KMGTP]?i?B)/([\d.]+\s*[KMGTP]?i?B)", line)
    if m2:
        out["current_human"] = m2.group(1).strip()
        out["total_human"] = m2.group(2).strip()
    m3 = re.search(r"DL:([\d.]+\s*[KMGTP]?i?B)", line)
    if m3: out["speed_human"] = m3.group(1).strip()
    m4 = re.search(r"ETA:(\S+)", line)
    if m4: out["eta_raw"] = m4.group(1).strip()
    return out

def _eta_to_seconds(raw):
    if not raw or raw == "--" or raw == "0s": return 0
    total = 0
    for value, unit in re.findall(r"(\d+)([hms])", raw):
        v = int(value)
        if unit == "h": total += v * 3600
        elif unit == "m": total += v * 60
        else: total += v
    return total

def _download_aria2(url, output_path, file_name, current_idx, total_files):
    temp_dir = _create_temp_dir()
    target_basename = os.path.basename(output_path)
    temp_output = os.path.join(temp_dir, target_basename)

    # Tambahkan Referer & User-Agent biar ga diblokir Eporner
    referer = "https://www.eporner.com/"

    cmd = [
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--max-connection-per-server=16", "--console-log-level=notice",
        "--summary-interval=1", "--allow-overwrite=true", "--auto-file-renaming=false",
        f"--user-agent={UA_DEFAULT}", f"--referer={referer}",
        f"--out={target_basename}", f"--dir={temp_dir}", url,
    ]

    try:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        last_percent = -1
        for line in process.stdout:
            line = line.strip()
            if not line: continue
            parsed = _parse_aria2_line(line)
            if not parsed: continue
            pct = parsed.get("percent", 0)
            if pct == last_percent: continue
            last_percent = pct

            cur_h = parsed.get("current_human", "0 B")
            tot_h = parsed.get("total_human", "0 B")
            spd_h = parsed.get("speed_human", "0 B/s")
            eta = _eta_to_seconds(parsed.get("eta_raw", "0s"))

            _set_state(
                phase="downloading", current_file=file_name, current_index=current_idx, total_files=total_files,
                percent=pct, current_human=cur_h, total_human=tot_h, speed_human=spd_h, eta_seconds=eta
            )
            _post_update({
                "status": "downloading", "phase": "download", "file_name": file_name,
                "percent": pct, "current_human": cur_h, "total_human": tot_h, "speed_human": spd_h, "eta_seconds": eta
            })
        process.wait()

        if process.returncode != 0: return False, f"aria2c exit {process.returncode}"
        if not os.path.exists(temp_output): return False, "no file produced"

        moved, err = _move_final(temp_output, output_path)
        return moved, err
    except Exception as e:
        return False, str(e)[:200]
    finally:
        _cleanup(temp_dir)

def _upload_one_file(local_path, file_name, hf_token, repo_id, repo_type, folder, current_idx, total_files):
    try: file_size = os.path.getsize(local_path)
    except Exception: file_size = 0

    _set_state(
        phase="uploading", current_file=os.path.basename(local_path), current_index=current_idx, total_files=total_files,
        percent=0, current_human="0 B", total_human=_format_size(file_size), speed_human="0 B/s", eta_seconds=None,
        message="Uploading to Hugging Face...", error="",
    )
    _post_update({
        "status": "uploading", "phase": "upload", "save_status": "uploading", "file_name": file_name,
        "percent": 0, "current_human": "0 B", "total_human": _format_size(file_size), "speed_human": "0 B/s",
    })

    stop_flag = {"stop": False}
    def _animate():
        start_ts = time.time()
        last_ts = 0
        while not stop_flag["stop"]:
            time.sleep(0.5)
            if stop_flag["stop"]: break
            elapsed = time.time() - start_ts
            if elapsed < 1.5: continue
            now = time.time()
            if now - last_ts < 2: continue
            last_ts = now

            frac = min(1 - pow(0.5, elapsed / 30.0), 0.95)
            pct = int(frac * 100)
            cur = int(file_size * frac)
            spd = cur / elapsed if elapsed > 0 else 0
            eta = int((file_size - cur) / spd) if spd > 0 and cur < file_size else None

            _set_state(phase="uploading", percent=pct, current_human=_format_size(cur), total_human=_format_size(file_size), speed_human=_format_size(spd) + "/s", eta_seconds=eta, message="")
            _post_update({
                "status": "uploading", "phase": "upload", "save_status": "uploading", "file_name": file_name,
                "percent": pct, "current_human": _format_size(cur), "total_human": _format_size(file_size),
                "speed_human": _format_size(spd) + "/s", "eta_seconds": eta,
            })

    threading.Thread(target=_animate, daemon=True).start()

    try:
        from huggingface_hub import HfApi
        api = HfApi(token=hf_token)
        path_in_repo = f"{folder}/{os.path.basename(local_path)}" if folder else os.path.basename(local_path)
        api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo, repo_id=repo_id, repo_type=repo_type, commit_message=f"Add {os.path.basename(path_in_repo)}")
        stop_flag["stop"] = True
        prefix = {"dataset": "datasets/", "space": "spaces/", "model": "models/"}.get(repo_type, "")
        save_url = f"https://huggingface.co/{prefix}{repo_id}/blob/main/{path_in_repo}"
        try:
            if os.path.exists(local_path): os.remove(local_path)
        except Exception: pass
        _set_state(phase="uploading", percent=100, current_human=_format_size(file_size), total_human=_format_size(file_size), speed_human="0 B/s", eta_seconds=0, message="Uploaded")
        _post_update({
            "status": "saved", "save_status": "done", "save_url": save_url, "file_name": file_name,
            "phase": "upload", "percent": 100, "current_human": _format_size(file_size), "total_human": _format_size(file_size), "speed_human": "0 B/s",
        })
        return True, save_url
    except Exception as e:
        stop_flag["stop"] = True
        _post_update({"status": "error", "save_status": "error", "save_error": str(e)[:300], "file_name": file_name, "phase": "upload"})
        return False, None

def _start_session(session_id, backend_url):
    global SESSION_ID, BACKEND_URL, COLAB_CONFIG, _ui_display

    session_id = (session_id or "").strip()
    backend_url = (backend_url or "").strip()

    if not session_id or not backend_url:
        _set_state(phase="error", error="Session ID and Backend URL are required")
        return

    SESSION_ID = session_id
    BACKEND_URL = backend_url
    _ui_display = None

    _set_state(phase="fetching", message=f"Fetching config from {backend_url}", error="")
    cfg = _fetch_config(session_id, backend_url)
    if not cfg or not cfg.get("ok"):
        _set_state(phase="error", error="Failed to fetch config. Check Session ID / Backend URL.")
        return

    COLAB_CONFIG = cfg
    url = cfg.get("url")

    url_lower = url.lower().split("?")[0].split("#")[0]
    is_direct = any(url_lower.endswith(ext) for ext in DIRECT_EXTS)

    formats = []

    if is_direct:
        fname = os.path.basename(url_lower) or "download"
        formats.append({"index": 1, "name": fname, "size": 0, "format_id": "direct"})
    else:
        _set_state(phase="waiting", message="Analyzing source URL...")
        info = _ytdlp_probe(url)
        if not info:
            _set_state(phase="error", error="Failed to extract. Blocked or not supported.")
            _post_update({"status": "error", "phase": "resolve", "error": "Failed to extract.", "finished": True})
            return

        title = info.get("title") or "download"
        idx = 1
        for f in info.get("formats", []):
            height = f.get("height")
            vcodec = (f.get("vcodec") or "none").lower()
            acodec = (f.get("acodec") or "none").lower()
            if vcodec == "none" and acodec == "none": continue
            if height and vcodec != "none":
                formats.append({"index": idx, "name": f"{title} [{height}p] ({f.get('ext', 'mp4')})", "size": f.get("filesize") or f.get("filesize_approx") or 0, "format_id": f.get("format_id")})
                idx += 1
            elif acodec != "none" and vcodec == "none":
                formats.append({"index": idx, "name": f"{title} [Audio] ({f.get('ext', 'm4a')})", "size": f.get("filesize") or f.get("filesize_approx") or 0, "format_id": f.get("format_id")})
                idx += 1

        if not formats:
            formats.append({"index": 1, "name": f"{title} [Best Auto]", "size": 0, "format_id": "best"})

    file_list = []
    for f in formats:
        file_list.append({"name": f.get("name", ""), "size": f.get("size", 0), "_status": "pending", "_index": f.get("index")})

    _state["files"] = file_list
    _set_state(phase="waiting", total_files=len(formats), message="Pilih format di web, Colab akan otomatis lanjut.")
    _post_update({"status": "waiting", "phase": "resolve", "finished": False, "formats": formats})

    while True:
        time.sleep(3)
        status = _check_status(SESSION_ID, BACKEND_URL)
        if status == "download":
            break
        elif status in ["cancel", "error"]:
            _set_state(phase="error", error="Dibatalkan oleh user.")
            return

    cfg = _fetch_config(SESSION_ID, BACKEND_URL)
    if not cfg or not cfg.get("formats"):
        _set_state(phase="error", error="Gagal mengambil daftar pilihan dari web.")
        return

    selected_formats = cfg.get("formats")
    hf_token = cfg.get("token")
    repo_id = cfg.get("repo_id")
    repo_type = cfg.get("repo_type") or "dataset"
    folder = cfg.get("folder") or ""

    total_files = len(selected_formats)
    completed = 0
    failed = 0

    _set_state(phase="downloading", total_files=total_files, completed_files=0, failed_files=0, message=f"Mulai download {total_files} file.", error="")

    downloaded_paths = []

    from yt_dlp import YoutubeDL
    for idx, item in enumerate(selected_formats, start=1):
        fname = _safe_filename(item.get("name", "download"), "")
        ext = fname.split('.')[-1] if '.' in fname else 'mp4'
        fmt_id = item.get("format_id")

        _state["files"][idx - 1]["_status"] = "downloading"
        _set_state(current_file=fname, current_index=idx)
        _post_update({"status": "downloading", "phase": "download", "file_name": fname, "percent": 0, "current_human": "0 B", "speed_human": "0 B/s"})

        temp_out = tempfile.mktemp(suffix=f".{ext}", prefix="uvd_video_", dir=TEMP_PATH)

        download_success = False
        err_msg = ""

        if fmt_id == "direct":
            _set_state(message="Downloading via aria2c (Fast)...")
            ok, err = _download_aria2(url, temp_out, fname, idx, total_files)
            if ok: download_success = True
            else: err_msg = err
        else:
            try:
                temp_dir = _create_temp_dir()
                target_basename = os.path.basename(temp_out)
                temp_output_ytdlp = os.path.join(temp_dir, target_basename)

                def _hook(d):
                    if d['status'] == 'downloading':
                        pct_str = d.get('_percent_str', '0%').replace('%','')
                        try: pct_val = float(pct_str)
                        except: pct_val = 0
                        spd_str = d.get('_speed_str', '0 B/s')
                        cur_str = d.get('_downloaded_bytes_str', '0 B')
                        tot_str = d.get('_total_bytes_str', '0 B')
                        eta = d.get('eta')

                        _set_state(percent=pct_val, current_human=cur_str, total_human=tot_str, speed_human=spd_str, eta_seconds=eta)
                        _post_update({"status": "downloading", "phase": "download", "file_name": fname, "percent": pct_val, "current_human": cur_str, "total_human": tot_str, "speed_human": spd_str, "eta_seconds": eta})

                opts = {
                    "outtmpl": temp_output_ytdlp, "format": fmt_id if fmt_id != "direct" else "best",
                    "merge_output_format": "mp4",
                    "quiet": True, "no_warnings": True, "noprogress": True,
                    "fragment_retries": 30, "retries": 30, "socket_timeout": 60, "concurrent_fragment_downloads": 32,
                    "http_chunk_size": 20 * 1024 * 1024, "ffmpeg_location": shutil.which("ffmpeg"),
                    "paths": {"temp": temp_dir}, "noplaylist": True, "progress_hooks": [_hook],
                }

                try:
                    with YoutubeDL(opts) as ydl: ydl.download([url])
                    actual = temp_output_ytdlp
                    if not os.path.exists(actual):
                        base_noext, _ = os.path.splitext(temp_output_ytdlp)
                        for e in [".mp4", ".mkv", ".webm", ".ts"]:
                            if os.path.exists(base_noext + e):
                                actual = base_noext + e; break
                    if os.path.exists(actual):
                        moved, _ = _move_final(actual, temp_out)
                        if moved: download_success = True
                except Exception as e:
                    err_msg = str(e)[:200]
                finally:
                    _cleanup(temp_dir)

            except Exception as e:
                err_msg = str(e)[:200]

        if download_success:
            _state["files"][idx - 1]["_status"] = "downloaded"
            downloaded_paths.append({"idx": idx, "file_name": fname, "path": temp_out, "size": os.path.getsize(temp_out)})
        else:
            failed += 1
            _state["files"][idx - 1]["_status"] = "error"
            _set_state(failed_files=failed, error=f"[{idx}/{total_files}] Gagal: {err_msg}")
            _post_update({"status": "error", "phase": "download", "error": err_msg, "finished": True})
            return

    _set_state(phase="uploading", completed_files=0, message=f"Mulai upload {len(downloaded_paths)} file.")
    for item in downloaded_paths:
        idx = item["idx"]
        fname = item["file_name"]
        fpath = item["path"]

        _state["files"][idx - 1]["_status"] = "uploading"
        _set_state(current_file=fname, current_index=idx)

        ok, _ = _upload_one_file(fpath, fname, hf_token, repo_id, repo_type, folder, idx, total_files)
        if ok:
            completed += 1
            _state["files"][idx - 1]["_status"] = "done"
        else:
            failed += 1
            _state["files"][idx - 1]["_status"] = "error"
        _set_state(completed_files=completed, failed_files=failed)

    final_phase = "done" if failed == 0 else "error"
    _set_state(phase=final_phase, percent=100, message=f"Selesai: {completed} OK, {failed} Gagal.")
    _post_update({"status": "done" if failed == 0 else "error", "phase": "done", "finished": True, "percent": 100})

def _on_start_click(session_id, backend_url):
    _start_session(session_id, backend_url)

try:
    from google.colab import output as _colab_output
    _colab_output.register_callback('universal_start', _on_start_click)
except Exception:
    pass

def _bootstrap():
    _ensure_dirs()

    form_html = """
    <div style="font-family: monospace; margin: 10px 0; padding: 18px;
                background: linear-gradient(135deg, #0f0f1e, #1a1a2e, #16213e);
                border-radius: 12px; border: 1px solid #2a2a4a; color: #e0e0e0;">
        <div style="color: #4CAF50; font-size: 18px; font-weight: bold;
                    letter-spacing: 2px; text-align: center;">
            UNIVERSAL COLAB
        </div>
        <div style="color: #888; font-size: 12px; margin-top: 4px;
                    text-align: center;">
            Download media universal di Colab · Upload otomatis ke Hugging Face
        </div>
        <div style="margin-top: 18px;">
            <label style="font-size: 11px; color: #888; letter-spacing: 1px;">SESSION ID</label>
            <input id="uvd-session-id" type="text" placeholder="paste dari kernel log"
                   style="width: 100%; margin-top: 6px; padding: 10px 12px; background: #0a0a14;
                          border: 1px solid #2a2a4a; border-radius: 6px; color: #e0e0e0;
                          font-family: monospace; font-size: 13px; outline: none; box-sizing: border-box;" />
        </div>
        <div style="margin-top: 12px;">
            <label style="font-size: 11px; color: #888; letter-spacing: 1px;">BACKEND URL</label>
            <input id="uvd-backend-url" type="text" value="https://warthadev-backend.onrender.com"
                   style="width: 100%; margin-top: 6px; padding: 10px 12px; background: #0a0a14;
                          border: 1px solid #2a2a4a; border-radius: 6px; color: #e0e0e0;
                          font-family: monospace; font-size: 13px; outline: none; box-sizing: border-box;" />
        </div>
        <button id="uvd-start-btn"
                style="margin-top: 16px; width: 100%; padding: 12px 20px;
                       background: linear-gradient(90deg, #4CAF50, #00BCD4);
                       border: none; border-radius: 8px; color: white;
                       font-family: monospace; font-size: 14px; font-weight: bold;
                       cursor: pointer; letter-spacing: 1px;">
            START DOWNLOAD
        </button>
    </div>
    """
    js_html = """
    <script>
    (function() {
        var btn = document.getElementById('uvd-start-btn');
        if (!btn) return;
        btn.addEventListener('click', function() {
            var sid = (document.getElementById('uvd-session-id').value || '').trim();
            var backend = (document.getElementById('uvd-backend-url').value || '').trim();
            if (!sid) { alert('Isi Session ID dulu'); return; }
            google.colab.kernel.invokeFunction('universal_start', [sid, backend], {});
        });
    })();
    </script>
    """
    display(HTML(form_html))
    display(HTML(js_html))

_bootstrap()